# Subsampling

See the corresponding documentation on [jinkommunity](https://community.jinko.ai/t/h7h4gf5/how-to-subsample-a-virtual-population)

In [1]:
# Cookbook specifics imports
import io
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats
import textwrap

# Jinko specifics imports & initialization
# Please fold this section and do not edit it
import jinko_helpers as jinko

# Connect to Jinko (see README.md for more options)
jinko.initialize()

## Step 0: Select trial of interest

In [2]:
"""
trial_short_id can be retrieved from the URL of your trial in Jinko, pattern is `https://jinko.ai/<trial_short_id>`
"""

trial_short_id = "tr-dI79-x7V2"

# folder ID, pattern is `https://jinko.ai/project/<project_id>?labels=<folder_id>`
# This folder is where the subsampling designs and the subsampled Vpops will be saved,
# it does not have to be the same folder as that of the initial trial
folder_id = "8764d184-3a1f-4524-90bc-49af9e640bdb"

## Step 1: Pick a trial version

In [3]:
# Choose a specific revision. By default we return the last version
revision = 26
# Choose a specific label. By default we return the last version
label = None
response = jinko.get_project_item(sid=trial_short_id, revision=revision, label=label)
trial_core_item_id, trial_snapshot_id = (
    response["coreId"]["id"],
    response["coreId"]["snapshotId"],
)

# # Uncomment the following if you want to use the latest completed or stopped version
# response = jinko.get_latest_calib_with_status(shortId=trial_short_id, statuses=["completed", "stopped"])
# trial_core_item_id, trial_snapshot_id = response["coreItemId"], response["snapshotId"]

print(
    f"Picked Trial with coreItemId: {trial_core_item_id}, snapshotId: {trial_snapshot_id}"
)
trial_link = jinko.get_project_item_url_from_sid(trial_short_id)
trial_link_with_revision = (
    f"{trial_link}?revision={revision}" if revision else trial_link
)
print(f"Trial link: {trial_link_with_revision}")

Picked Trial with coreItemId: 335c7d37-d9fb-4b5a-be31-78a7c4a9ea91, snapshotId: 60354678-eab3-4074-8381-e7276cc4f3c6
Trial link: https://jinko.ai/tr-dI79-x7V2?revision=26


## Step 2: Define the subsampling design

In [4]:
vpop_generator_payload = {
    "contents": {
        "trialId": {
            "coreItemId": trial_core_item_id,
            "snapshotId": trial_snapshot_id,
        },
        "filters": [
            {
                "tag": "DescriptorFilter",
                "contents": [
                    {
                        "descriptorId": "isResponse.tend",
                        "arm": "DoubleDose",
                        "operator": "Gte",
                        "value": 0.5,
                    }
                ],
            },
            {
                "tag": "CategoricalDescriptorFilter",
                "contents": [
                    {
                        "descriptorId": "sex",
                        "arm": "crossArms",
                        "levels": ["female"],
                    }
                ],
            },
        ],
        "targetMarginals": [
            {
                "arm": "SingleDose",
                "distribution": {
                    "tag": "LogNormal",
                    "mean": -4,
                    "stdev": 0.1,
                    "base": 10,
                },
                "id": "Blood.Drug-avg-from-PT0S-to-P100D",
                "weight": 1,
            },
            {
                "arm": "DoubleDose",
                "distribution": {"tag": "Uniform", "lowBound": 2e-4, "highBound": 4e-4},
                "id": "Blood.Drug-avg-from-PT0S-to-P100D",
                "weight": 1,
            },
        ],
        "targetCategoricals": [
            {
                "id": "origin",
                "arm": "crossArms",
                "distribution": {
                    "tag": "Categorical",
                    "catMapping": {
                        "EU": 0.6,
                        "APAC": 0.2,
                        "US": 0.2,
                    },
                },
                "weight": 1,
            }
        ],
        "targetCorrelations": [
            {
                "correlateX": {
                    "arm": None,
                    "id": "kClearanceDrug.tmin",
                },
                "correlateY": {
                    "arm": "DoubleDose",
                    "id": "tumorBurden.tend",
                },
                "correlationCoefficient": 0.5,
                "weight": 0.1,
            }
        ],
        "targetSurvivals": [
            {
                "timeToEventScalarId": "timeOfTumorReduction",
                "arm": "SingleDose",
                "timeVals": [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
                "cumulativeSurvivalRates": [
                    0.9,
                    0.9,
                    0.6,
                    0.5,
                    0.4,
                    0.2,
                    0.15,
                    0.1,
                    0.05,
                    0,
                ],
                "timeUnit": "day",
                "weight": 1,
            }
        ],
        "targetSummaryStatistics": [
            {
                "arm": "Control",
                "id": "Tumor.CancerCell-at-P100D",
                "mean": 1e11,
                "standardDeviation": 2e10,
                "weight": 1,
            }
        ],
        "additionalScalars": [
            {"id": "tumorBurden.tend", "arm": "DoubleDose"},
            {"id": "kClearanceDrug.tmin", "arm": None},
            {"id": "timeOfTumorReduction", "arm": "SingleDose"},
            {"id": "Tumor.CancerCell-at-P100D", "arm": "Control"},
            {"id": "origin", "arm": None},
            {"id": "sex", "arm": None},
        ],
    },
    "tag": "FromSubsamplingGenerator",
}

if "subsampling_core_item_id" in globals() and "subsampling_snapshot_id" in globals():
    response = jinko.make_request(
        path=f"/core/v2/vpop_manager/vpop_generator/{subsampling_core_item_id}",
        method="PUT",
        json=vpop_generator_payload,
    )
    project_item_info = jinko.get_project_item_info_from_response(response)
    subsampling_core_item_id = project_item_info["coreItemId"]["id"]
    subsampling_snapshot_id = project_item_info["coreItemId"]["snapshotId"]
    subsampling_url = jinko.get_project_item_url_from_response(response)
    print(f"Subsampling design: {subsampling_url}")
else:
    response = jinko.make_request(
        path="/core/v2/vpop_manager/vpop_generator",
        method="POST",
        json=vpop_generator_payload,
        options={"name": "Subsampling Design", "folder_id": folder_id},
    )
    project_item_info = jinko.get_project_item_info_from_response(response)
    subsampling_core_item_id = project_item_info["coreItemId"]["id"]
    subsampling_snapshot_id = project_item_info["coreItemId"]["snapshotId"]
    subsampling_url = jinko.get_project_item_url_from_response(response)
    print(f"Subsampling design: {subsampling_url}")

Subsampling design: https://jinko.ai/sd-IAt0-pJeT


## Step 3: Run the subsampling design

In [5]:
# Set the subsampling options
subsampling_payload = {
    "tag": "VpopGeneratorOptionsForSubsampling",
    "contents": {
        "numSamples": 50,
        "seed": 0,
        "numIterations": 10000,
        "itersFixedTemperature": 20,
        "replacementRate": 0.01,
        "boltzmannConstant": 0.001,
    },
}

response = jinko.make_request(
    path=f"/core/v2/vpop_manager/vpop_generator/{subsampling_core_item_id}/snapshots/{subsampling_snapshot_id}/vpop",
    method="POST",
    json=subsampling_payload,
    options={"name": f"Subsampled Vpop", "folder_id": folder_id},
)
subsampled_vpop_core_item_id = response.json()["coreItemId"]
fitness = response.json()["subsamplingFitness"]
print(f"Subsampled vpop: {jinko.get_project_item_url_from_response(response)}")
goodness_dataframe = jinko.subsampling_goodness_of_fit_as_dataframe(fitness)
display(goodness_dataframe)
print(f"Weighted score = {fitness["weightedScore"]:.3g}")

Subsampled vpop: https://jinko.ai/vp-M7rT-ytvA


,qoi,arm,score,targetType,qoi2,arm2
0,Blood.Drug-avg-from-PT0S-to-P100D,DoubleDose,0.735525,marginal,NaN,NaN
1,Blood.Drug-avg-from-PT0S-to-P100D,SingleDose,0.640948,marginal,NaN,NaN
2,origin,crossArms,1.000000,categorical,NaN,NaN
3,timeOfTumorReduction,SingleDose,0.900000,survival,NaN,NaN
4,Tumor.CancerCell-at-P100D,Control,0.977727,summary statistics,NaN,NaN
5,kClearanceDrug.tmin,None,0.760183,correlation,tumorBurden.tend,DoubleDose


Weighted score = 0.849


## Step 5: Compare the initial and subsampled vpops

In [6]:
initial_trial = jinko.make_request(
    path=f"/core/v2/trial_manager/trial/{trial_core_item_id}/snapshots/{trial_snapshot_id}",
    method="GET",
).json()
initial_vpop_core_item_id = initial_trial["vpopId"]["coreItemId"]

initial_vpop = jinko.make_request(
    path=f"/core/v2/vpop_manager/vpop/{initial_vpop_core_item_id}",
    method="GET",
    options={"output_format": "text/csv"},
)
initial_vpop_df = pd.read_csv(io.StringIO(str(initial_vpop.content, "utf-8")))

# Get the subsampled Vpop
subsampled_vpop = jinko.make_request(
    path=f"/core/v2/vpop_manager/vpop/{subsampled_vpop_core_item_id}",
    method="GET",
    options={"output_format": "text/csv"},
)
subsampled_vpop_df = pd.read_csv(io.StringIO(str(subsampled_vpop.content, "utf-8")))

In [11]:
# Add here the parameters for which you want the plots to be in log scale
logParams = [
    "kClearanceDrug",
]

colnames = list(c for c in initial_vpop_df.columns if c != "patientIndex")
dimension = len(colnames)
colnames_wrapped = ["<br>".join(textwrap.wrap(t, width=30)) for t in colnames]
num_rows, num_cols = 3, 4
if num_rows * num_cols < dimension:
    raise Exception(
        f"Not enough rows and columns ({num_rows * num_cols=}) to plot {dimension} parameters"
    )
nbinsx = 21
subsampled_vpop_color = "#330C73"
initial_vpop_color = "#EB89B5"
def ij_to_k(i, j):
    return num_cols * i + j


fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    horizontal_spacing=0.01,
    vertical_spacing=0.1,
    subplot_titles=colnames_wrapped,
)
show_legend_log, show_legend_linear = True, True
for i in range(num_rows):
    for j in range(num_cols):
        k = ij_to_k(i, j)
        if k >= dimension:
            continue
        else:
            col_name = colnames[k]
            if col_name in logParams:
                init_x = np.log10(initial_vpop_df[col_name])
                subsampled_x = np.log10(subsampled_vpop_df[col_name])
                legend_group, legendgrouptitle_text = "log", "Log10-transformed"
                marker_color_init, marker_color_subsampled = "#89EBBF", "#73330C"
                showlegend = show_legend_log
            else:
                init_x = initial_vpop_df[col_name]
                subsampled_x = subsampled_vpop_df[col_name]
                legend_group, legendgrouptitle_text = "linear", "Linear scale"
                marker_color_init, marker_color_subsampled = initial_vpop_color, subsampled_vpop_color
                showlegend = show_legend_linear

            ksresults = stats.ks_2samp(init_x, subsampled_x)
            fig.add_trace(
                go.Histogram(
                    name="Initial",
                    x=init_x,
                    histnorm="probability density",
                    nbinsx=nbinsx,
                    marker_color=marker_color_init,
                    legendgroup=legend_group,
                    legendgrouptitle_text=legendgrouptitle_text,
                    showlegend=showlegend,
                ),
                row=i + 1,
                col=j + 1,
            )
            fig.add_trace(
                go.Histogram(
                    name="Subsampled",
                    x=subsampled_x,
                    histnorm="probability density",
                    nbinsx=nbinsx,
                    marker_color=marker_color_subsampled,
                    legendgroup=legend_group,
                    showlegend=showlegend,
                    legendgrouptitle_text=legendgrouptitle_text,
                ),
                row=i + 1,
                col=j + 1,
            )
            fig.layout.annotations[
                k
            ].text += f"<br>KS={ksresults.statistic:.2f}, p={ksresults.pvalue:.2e}"
            if col_name in logParams:
                show_legend_log = False
                fig.update_xaxes(tickprefix="1e", row=i + 1, col=j + 1)
            else:
                show_legend_linear = False

fig.update_annotations(font_size=14)
figure_width = 1200
figure_height = 1000
fig.update_layout(
    font=dict(size=12),
    showlegend=True,
    bargap=0.15,
    width=figure_width,
    height=figure_height,
    template="plotly_white",
)
config = {
    "toImageButtonOptions": {
        "format": "png",  # one of png, svg, jpeg, webp
        "filename": "custom_image",
        "height": figure_height,
        "width": figure_width,
        "scale": 6,  # Multiply title/legend/axis/canvas sizes by this factor
    }
}
fig.show(config=config)

In [12]:
trial_summary = jinko.get_trial_scalars_summary(
    trial_core_item_id, trial_snapshot_id, print_summary=False
)
print("Available scalars (showing first 10 only):")
for scalar in trial_summary["scalars"][:10]:
    print(f"  {scalar["id"]} on arm(s) {scalar["arms"]}")
print("Available cross-arm scalars (showing first 10 only):")
for scalar in trial_summary["scalarsCrossArm"][:10]:
    print(f"  {scalar["id"]} on arm 'crossArms'")

Available scalars (showing first 10 only):
  Blood.Drug-avg-from-PT0S-to-P100D on arm(s) ['Control', 'DoubleDose', 'SingleDose']
  Lymph.Drug.auc on arm(s) ['Control', 'DoubleDose', 'SingleDose']
  Lymph.Drug.max on arm(s) ['Control', 'DoubleDose', 'SingleDose']
  Lymph.Drug.mean on arm(s) ['Control', 'DoubleDose', 'SingleDose']
  Lymph.Drug.min on arm(s) ['Control', 'DoubleDose', 'SingleDose']
  Lymph.Drug.tend on arm(s) ['Control', 'DoubleDose', 'SingleDose']
  Lymph.Drug.timeofmax on arm(s) ['Control', 'DoubleDose', 'SingleDose']
  SimulationTMax on arm(s) ['Control', 'DoubleDose', 'SingleDose']
  SimulationTMin on arm(s) ['Control', 'DoubleDose', 'SingleDose']
  SubcutaneousInjectionSite.Drug.auc on arm(s) ['Control', 'DoubleDose', 'SingleDose']
Available cross-arm scalars (showing first 10 only):
  Lymph.Drug.tmin on arm 'crossArms'
  SubcutaneousInjectionSite.Drug.tmin on arm 'crossArms'
  Tissue.Drug.tmin on arm 'crossArms'
  Tumor.CancerCell.tmin on arm 'crossArms'
  Tumor.Drug

In [13]:
# list here the quantities of interest for which you want to plot the histograms and CDFs in initial Vpop vs subsampled Vpop
# this comes in the form of a map from scalar ID to protocol arms
scalar_ids = {
    ("Blood.Drug-avg-from-PT0S-to-P100D", "DoubleDose"),
    ("initialTumorBurden.tmin", "crossArms"),
}
# this gets the requested scalar IDs on ALL arms
scalars_all_arms_df = jinko.get_trial_scalars_as_dataframe(
    trial_core_item_id,
    trial_snapshot_id,
    list(scalar for scalar, _ in scalar_ids),
)
# Filter the dataframe to only keep the (scalar ID, arm ID) pairs we are interested in
scalars_df = scalars_all_arms_df.loc[
    scalars_all_arms_df[["scalarId", "armId"]].apply(tuple, axis=1).isin(scalar_ids)
].copy()

display(scalars_df)

,patientId,armId,scalarId,value,unit
1000,004100bf-0b67-52e3-a2ea-0b965cbed55a,DoubleDose,Blood.Drug-avg-from-PT0S-to-P100D,0.000203,mg / mL
1001,004d901d-c7a9-94a5-3d7e-109e8b7ce41a,DoubleDose,Blood.Drug-avg-from-PT0S-to-P100D,0.000275,mg / mL
1002,00830fa7-94a9-a060-9058-14fc20b40895,DoubleDose,Blood.Drug-avg-from-PT0S-to-P100D,0.000287,mg / mL
1003,00b774c0-a5d7-f1a8-c8b0-ac8fb5f60851,DoubleDose,Blood.Drug-avg-from-PT0S-to-P100D,0.000117,mg / mL
1004,012b0617-9897-8481-cae3-7dc6641e9817,DoubleDose,Blood.Drug-avg-from-PT0S-to-P100D,0.000109,mg / mL
...,...,...,...,...,...
3987,ffb45283-b7cd-efb1-bd21-d7a53e7eebb2,crossArms,initialTumorBurden.tmin,36.729055,cm ** 3
3988,ffb88fd3-88f7-d7f0-85ee-34c7f10ffac0,crossArms,initialTumorBurden.tmin,32.981848,cm ** 3
3989,ffce5d42-2fff-c4ef-f27f-939d1fc108a8,crossArms,initialTumorBurden.tmin,20.379245,cm ** 3
3990,ffceda1e-c954-5311-3cc6-4cf6f9467247,crossArms,initialTumorBurden.tmin,34.744423,cm ** 3


In [14]:
# set here the theoretical distributions for each (scalar ID, arm ID) pair
theoretical_distributions = {
    ("Blood.Drug-avg-from-PT0S-to-P100D", "DoubleDose"): stats.norm(
        loc=0.0002, scale=0.0001
    ),
    ("initialTumorBurden.tmin", "crossArms"): stats.norm(loc=50, scale=10),
}

# Keep only scalar rows for patients present in the subsampled vpop
subsampled_patient_ids = set(subsampled_vpop_df["patientIndex"].astype(str))
subsampled_scalars_df = scalars_df.loc[
    scalars_df["patientId"].isin(subsampled_patient_ids)
].copy()

dimension = len(scalar_ids)
if dimension == 0:
    raise ValueError("scalar_ids is empty.")


def normal_overlay_grid(series_a, series_b):
    return np.linspace(
        min(series_a.min(), series_b.min()), max(series_a.max(), series_b.max()), 400
    )


nbinsx = 21
subplot_titles = [f"{sid}<br>{arm}<br>Histogram" for sid, arm in scalar_ids] + [
    f"{sid}<br>{arm}<br>CDF" for sid, arm in scalar_ids
]

fig = make_subplots(
    rows=2,
    cols=dimension,
    horizontal_spacing=min(0.08, 0.2 / dimension),
    vertical_spacing=0.2,
    subplot_titles=subplot_titles,
)

initial_n_patients = scalars_df["patientId"].nunique()
subsampled_n_patients = subsampled_scalars_df["patientId"].nunique()

for col_idx, (scalar_id, arm_id) in enumerate(scalar_ids, start=1):
    initial_x = pd.to_numeric(
        scalars_df.loc[
            (scalars_df["scalarId"] == scalar_id) & (scalars_df["armId"] == arm_id),
            "value",
        ],
        errors="coerce",
    )
    subsampled_x = pd.to_numeric(
        subsampled_scalars_df.loc[
            (subsampled_scalars_df["scalarId"] == scalar_id)
            & (subsampled_scalars_df["armId"] == arm_id),
            "value",
        ],
        errors="coerce",
    )
    fig.add_trace(
        go.Histogram(
            name=f"Initial (N={initial_n_patients})",
            x=initial_x,
            histnorm="probability density",
            nbinsx=nbinsx,
            marker_color=initial_vpop_color,
            opacity=0.65,
            legendgroup="initial",
            showlegend=(col_idx == 1),
        ),
        row=1,
        col=col_idx,
    )

    fig.add_trace(
        go.Histogram(
            name=f"Subsampled (N={subsampled_n_patients})",
            x=subsampled_x,
            histnorm="probability density",
            nbinsx=nbinsx,
            marker_color=subsampled_vpop_color,
            opacity=0.65,
            legendgroup="subsampled",
            showlegend=(col_idx == 1),
        ),
        row=1,
        col=col_idx,
    )

    theoretical_x = normal_overlay_grid(initial_x, subsampled_x)
    theoretical_distribution = theoretical_distributions.get((scalar_id, arm_id))
    if theoretical_distribution is not None:
        fig.add_trace(
            go.Scatter(
                name="theoretical PDF",
                x=theoretical_x,
                y=theoretical_distribution.pdf(theoretical_x),
                mode="lines",
                line=dict(color="#1E1E1E", width=2, dash="dash"),
                legendgroup="normal_pdf",
                showlegend=(col_idx == 1),
            ),
            row=1,
            col=col_idx,
        )
        fig.add_trace(
            go.Scatter(
                name="theoretical CDF",
                x=theoretical_x,
                y=theoretical_distribution.cdf(theoretical_x),
                mode="lines",
                line=dict(color="#1E1E1E", width=2, dash="dot"),
                legendgroup="normal_cdf",
                showlegend=(col_idx == 1),
            ),
            row=2,
            col=col_idx,
        )

    ecdf = stats.ecdf(initial_x.to_numpy())
    fig.add_trace(
        go.Scatter(
            name="Initial CDF",
            x=ecdf.cdf.quantiles,
            y=ecdf.cdf.probabilities,
            mode="lines",
            line=dict(color=initial_vpop_color, width=2),
            legendgroup="initial_cdf",
            showlegend=(col_idx == 1),
        ),
        row=2,
        col=col_idx,
    )

    subsampled_ecdf = stats.ecdf(subsampled_x.to_numpy())
    fig.add_trace(
        go.Scatter(
            name="Subsampled CDF",
            x=subsampled_ecdf.cdf.quantiles,
            y=subsampled_ecdf.cdf.probabilities,
            mode="lines",
            line=dict(color=subsampled_vpop_color, width=2),
            legendgroup="subsampled_cdf",
            showlegend=(col_idx == 1),
        ),
        row=2,
        col=col_idx,
    )

for col_idx in range(1, dimension + 1):
    fig.update_yaxes(range=[0, 1], row=2, col=col_idx)
fig.update_yaxes(title_text="Density", row=1, col=1)
fig.update_yaxes(title_text="Cumulative probability", row=2, col=1)

fig.update_annotations(font_size=14)
figure_width = max(1200, 450 * dimension)
figure_height = 900
fig.update_layout(
    font=dict(size=12),
    showlegend=True,
    bargap=0.15,
    barmode="overlay",
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
    width=figure_width,
    height=figure_height,
    template="plotly_white",
)
config = {
    "toImageButtonOptions": {
        "format": "png",
        "filename": "scalar_histograms_and_cdfs",
        "height": figure_height,
        "width": figure_width,
        "scale": 6,
    }
}
fig.show(config=config)